In [1]:
%%pyspark default.spark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator


Executing for connection type: SPARK_GLUE, connection name: default.spark
Creating Glue session...
Create session for connection: default.spark


'Session 6ta2h2hm7s42zr-2fe9ebf8-5d47-4b86-a907-a7f84eaa71b9 has been created.'

<sagemaker_studio_dataengineering_sessions.sagemaker_base_session_manager.common.debugging_utils.SessionInfoTableDisplay object>

Session created for connection: default.spark.

Connection: default.spark | Run start time: 2026-07-28 12:32:09.494405 | Run duration : 0:01:20.590135s.


In [2]:
%%pyspark default.spark
path = "s3://yelpdatasetvita/gold_layer/ml/recommendation_features/"

df = spark.read.parquet(path)



Connection: default.spark | Run start time: 2026-07-28 12:33:30.091016 | Run duration : 0:00:09.670473s.


In [3]:
%%pyspark default.spark

print("Number of Rows :", df.count())

Number of Rows : 6990280

Connection: default.spark | Run start time: 2026-07-28 12:36:04.002042 | Run duration : 0:00:15.027507s.


In [4]:
%%pyspark default.spark
print("Columns :")
print(df.columns)


Columns :
['user_id', 'business_id', 'stars', 'interaction_recency', 'user_review_count', 'business_review_count']

Connection: default.spark | Run start time: 2026-07-28 12:36:42.218837 | Run duration : 0:00:07.956691s.


In [5]:
%%pyspark default.spark
from pyspark.sql.functions import when, count

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()


+-------+-----------+-----+-------------------+-----------------+---------------------+
|user_id|business_id|stars|interaction_recency|user_review_count|business_review_count|
+-------+-----------+-----+-------------------+-----------------+---------------------+
|      0|          0|    0|                  0|               33|                    0|
+-------+-----------+-----+-------------------+-----------------+---------------------+

Connection: default.spark | Run start time: 2026-07-28 12:37:04.448340 | Run duration : 0:00:15.314242s.


In [6]:
%%pyspark default.spark

df = df.dropDuplicates()


Connection: default.spark | Run start time: 2026-07-28 12:37:19.770913 | Run duration : 0:00:04.148978s.


In [10]:
%%pyspark default.spark


from pyspark.sql.functions import monotonically_increasing_id

# Create user index
users = (
    df.select("user_id")
      .distinct()
      .withColumn("userIndex", monotonically_increasing_id())
)

# Join back
df = df.join(users, "user_id")

# Create business index
business = (
    df.select("business_id")
      .distinct()
      .withColumn("businessIndex", monotonically_increasing_id())
)

df = df.join(business, "business_id")

df.show(5)

+--------------------+--------------------+-----+-------------------+-----------------+---------------------+------------+-------------+
|         business_id|             user_id|stars|interaction_recency|user_review_count|business_review_count|   userIndex|businessIndex|
+--------------------+--------------------+-----+-------------------+-----------------+---------------------+------------+-------------+
|--OS_I7dnABrXvRCC...|RAVr_68v9aXWs9haV...|  5.0|               3844|               19|                    5|292057776618|         1566|
|--OS_I7dnABrXvRCC...|sGRYJlEqvkeqNkwD-...|  5.0|               2645|               27|                    5| 77309416877|         1566|
|--OS_I7dnABrXvRCC...|Bc-aAEmS9FiDiJ6bt...|  1.0|               3687|                2|                    5|  8589958119|         1566|
|--OS_I7dnABrXvRCC...|DzWToz-VRSOGB6Cje...|  5.0|               2693|                3|                    5|103079231703|         1566|
|--OS_I7dnABrXvRCC...|WZpoceDdEd8BgEy8T..

In [12]:
%%pyspark default.spark
from pyspark.sql.functions import monotonically_increasing_id

# Remove old businessIndex if it exists
if "businessIndex" in df.columns:
    df = df.drop("businessIndex")

# Create mapping
business_mapping = (
    df.select("business_id")
      .distinct()
      .withColumn("businessIndex", monotonically_increasing_id())
)

# Join mapping back
df = df.join(business_mapping, on="business_id", how="left")

# Check result
df.select("business_id", "businessIndex").show(10, truncate=False)

+----------------------+-------------+
|business_id           |businessIndex|
+----------------------+-------------+
|--OS_I7dnABrXvRCCuWOGQ|4079         |
|--OS_I7dnABrXvRCCuWOGQ|4079         |
|--OS_I7dnABrXvRCCuWOGQ|4079         |
|--OS_I7dnABrXvRCCuWOGQ|4079         |
|--OS_I7dnABrXvRCCuWOGQ|4079         |
|-0H_3r6z2giieA6oSTUFKQ|4098         |
|-0H_3r6z2giieA6oSTUFKQ|4098         |
|-0H_3r6z2giieA6oSTUFKQ|4098         |
|-0H_3r6z2giieA6oSTUFKQ|4098         |
|-0H_3r6z2giieA6oSTUFKQ|4098         |
+----------------------+-------------+
only showing top 10 rows

Connection: default.spark | Run start time: 2026-07-28 12:47:28.236676 | Run duration : 0:00:18.408849s.


In [13]:
%%pyspark default.spark

from pyspark.sql.functions import col

df = df.withColumn(
    "userIndex",
    col("userIndex").cast("integer")
)

df = df.withColumn(
    "businessIndex",
    col("businessIndex").cast("integer")
)


Connection: default.spark | Run start time: 2026-07-28 12:48:49.320280 | Run duration : 0:00:04.021018s.


In [14]:
%%pyspark default.spark
ratings = df.select(
    col("userIndex"),
    col("businessIndex"),
    col("stars").alias("rating")
)



Connection: default.spark | Run start time: 2026-07-28 12:49:35.162757 | Run duration : 0:00:04.204679s.


In [15]:
%%pyspark default.spark
ratings.printSchema()

ratings.show(10, truncate=False)


root
 |-- userIndex: integer (nullable = false)
 |-- businessIndex: integer (nullable = true)
 |-- rating: double (nullable = true)

+---------+-------------+------+
|userIndex|businessIndex|rating|
+---------+-------------+------+
|17827    |2119         |5.0   |
|565      |2119         |1.0   |
|5163     |2119         |5.0   |
|43055    |2119         |5.0   |
|37975    |2119         |5.0   |
|9372     |3338         |1.0   |
|41305    |3338         |1.0   |
|10218    |3338         |5.0   |
|26011    |3338         |1.0   |
|1685     |3338         |4.0   |
+---------+-------------+------+
only showing top 10 rows

Connection: default.spark | Run start time: 2026-07-28 12:50:33.318716 | Run duration : 0:00:18.338407s.


In [16]:
%%pyspark default.spark
train, test = ratings.randomSplit([0.8, 0.2], seed=42)

print("Training Rows :", train.count())
print("Testing Rows :", test.count())


Training Rows : 5590130
Testing Rows : 1399080

Connection: default.spark | Run start time: 2026-07-28 12:51:15.882136 | Run duration : 0:00:30.114066s.


In [21]:
%%pyspark default.spark
from pyspark.ml.recommendation import ALS

als = ALS(
    userCol="userIndex",
    itemCol="businessIndex",
    ratingCol="rating",
    coldStartStrategy="drop",
    nonnegative=True,
    implicitPrefs=False,
    rank=20,
    maxIter=20,
    regParam=0.05
)



Connection: default.spark | Run start time: 2026-07-28 13:04:50.589365 | Run duration : 0:00:04.333610s.


In [22]:
%%pyspark default.spark

model = als.fit(train)


Connection: default.spark | Run start time: 2026-07-28 13:05:17.968456 | Run duration : 0:00:35.281860s.


In [23]:
%%pyspark default.spark
predictions = model.transform(test)

predictions.show(10, truncate=False)


+---------+-------------+------+----------+
|userIndex|businessIndex|rating|prediction|
+---------+-------------+------+----------+
|0        |68           |1.0   |3.6421418 |
|0        |89           |4.0   |4.1523695 |
|0        |100          |4.0   |4.121523  |
|0        |150          |5.0   |4.043493  |
|0        |214          |4.0   |3.7261128 |
|0        |289          |2.0   |3.7242334 |
|0        |455          |5.0   |4.1323705 |
|0        |562          |3.0   |3.7418642 |
|0        |950          |3.0   |3.726333  |
|0        |988          |5.0   |4.2407913 |
+---------+-------------+------+----------+
only showing top 10 rows

Connection: default.spark | Run start time: 2026-07-28 13:06:24.658946 | Run duration : 0:00:16.439892s.


In [24]:
%%pyspark default.spark
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

rmse = evaluator.evaluate(predictions)

print("RMSE:", rmse)


RMSE: 1.5211872573749343

Connection: default.spark | Run start time: 2026-07-28 13:08:18.018073 | Run duration : 0:00:18.331202s.


In [26]:
%%pyspark default.spark

from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

best_rmse = float("inf")
best_params = None

for rank in [10, 20, 30, 40, 50]:
    for reg in [0.01, 0.05, 0.1, 0.15]:
        als = ALS(
            userCol="userIndex",
            itemCol="businessIndex",
            ratingCol="rating",
            rank=rank,
            regParam=reg,
            maxIter=20,
            coldStartStrategy="drop"
        )

        model = als.fit(train)
        predictions = model.transform(test)
        rmse = evaluator.evaluate(predictions)

        print(f"Rank={rank}, Reg={reg}, RMSE={rmse:.4f}")

        if rmse < best_rmse:
            best_rmse = rmse
            best_params = (rank, reg)

print("\nBest RMSE:", best_rmse)
print("Best Parameters:", best_params)

Rank=10, Reg=0.01, RMSE=1.5840
Rank=10, Reg=0.05, RMSE=1.5124
Rank=10, Reg=0.1, RMSE=1.4817
Rank=10, Reg=0.15, RMSE=1.4705
Rank=20, Reg=0.01, RMSE=1.7650
Rank=20, Reg=0.05, RMSE=1.5611
Rank=20, Reg=0.1, RMSE=1.4935
Rank=20, Reg=0.15, RMSE=1.4716
Rank=30, Reg=0.01, RMSE=1.8779
Rank=30, Reg=0.05, RMSE=1.5648
Rank=30, Reg=0.1, RMSE=1.4908
Rank=30, Reg=0.15, RMSE=1.4843
Rank=40, Reg=0.01, RMSE=1.8899
Rank=40, Reg=0.05, RMSE=1.5485
Rank=40, Reg=0.1, RMSE=1.4917
Rank=40, Reg=0.15, RMSE=1.4704
Rank=50, Reg=0.01, RMSE=1.8783
Rank=50, Reg=0.05, RMSE=1.5570
Rank=50, Reg=0.1, RMSE=1.4774
Rank=50, Reg=0.15, RMSE=1.4719

Best RMSE: 1.4704014067077866
Best Parameters: (40, 0.15)

Connection: default.spark | Run start time: 2026-07-28 13:18:23.097316 | Run duration : 0:14:53.066329s.


In [29]:
%%pyspark default.spark
predictions.show(20, truncate=False)

+---------+-------------+------+----------+
|userIndex|businessIndex|rating|prediction|
+---------+-------------+------+----------+
|18       |189          |3.0   |3.9071577 |
|18       |523          |5.0   |3.5908923 |
|18       |672          |4.0   |3.907869  |
|18       |2216         |5.0   |3.5338788 |
|18       |2322         |2.0   |4.016143  |
|38       |21           |5.0   |3.6289408 |
|38       |56           |4.0   |3.2813737 |
|38       |120          |4.0   |3.653868  |
|38       |146          |5.0   |3.1260147 |
|38       |370          |3.0   |3.1941743 |
|38       |508          |4.0   |3.5157576 |
|38       |597          |5.0   |3.615722  |
|38       |716          |3.0   |3.219042  |
|38       |855          |5.0   |3.1009064 |
|38       |1061         |5.0   |3.1444473 |
|38       |2256         |2.0   |3.428123  |
|38       |3287         |5.0   |3.5784445 |
|46       |66           |4.0   |3.4943502 |
|46       |238          |3.0   |3.7677746 |
|46       |410          |2.0   |

In [31]:
%%pyspark default.spark
predictions = model.transform(test)

predictions.write \
    .mode("overwrite") \
    .parquet("s3://yelpdatasetvita/gold_layer/collabrative_filtering_output/predictions/")



Connection: default.spark | Run start time: 2026-07-28 13:56:53.209416 | Run duration : 0:00:20.406301s.


In [ ]:
%%pyspark default.spark

